# Module 2 · Lesson 05: Prompt Evaluation

How do you know if your prompt is **good**? In production, we use **LLM-as-Judge**
to automatically evaluate prompt quality.

## What you will learn
1. Why prompt evaluation matters
2. **LLM-as-Judge** — using one model to evaluate another
3. Building a **scoring rubric**
4. **A/B testing** prompts
5. Batch evaluation for consistency

In [9]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / "module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")

OpenAI client ready - using model gpt-4o-mini


In [10]:
def ask_open_ai(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model:  str="gpt-4o-mini") -> str:
    """
    Send a prompt to the OpenAI Chat Completions API and return the assistant's reply.

    This helper function builds a chat message list, optionally including a system
    instruction, sends it to the specified model, and returns the generated text
    content from the first response choice.

    Args:
        prompt (str): The user prompt to send to the model.
        system_content (str, optional): Optional system-level instruction that
            defines the assistant's behavior, tone, or constraints. If provided,
            it is added as the first message in the conversation. Defaults to None.
        temperature (float, optional): Sampling temperature used to control
            randomness in the model's response. Lower values produce more
            deterministic outputs; higher values produce more creative or varied
            outputs. Defaults to 0.7.
        max_resp_tokens (int, optional): Maximum number of tokens allowed in the
            generated response. Defaults to 800.
        ai_model (str, optional): The model name to use for the request.
            Defaults to "gpt-4o-mini".

    Returns:
        str: The text content of the assistant's first response message.

    Raises:
        Exception: Propagates any exception raised by the OpenAI client request
            (for example, authentication errors, rate limits, invalid parameters,
            or network-related failures).

    Example of use:
         reply = ask("Explain recursion in simple terms.")
         print(reply)

        reply = ask(
             prompt="Write a haiku about Python.",
             system_content="You are a poetic assistant.",
             temperature=0.9,
             max_resp_tokens=100,
             ai_model="gpt-4o-mini"
        )        
        print(reply)
    """
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

display(Markdown(f"**The function has run**"))

**The function has run**

### As judge we will use model from Antropic. 
  

In [11]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output
 
load_dotenv(Path.cwd().parent / "module_02_prompt_enginnering/.env")

from anthropic import Anthropic

claude_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
CLAUDE_MODEL = "claude-sonnet-4-6" 

if claude_client:
    print(f"Antropic client ready - using model {CLAUDE_MODEL}")

Antropic client ready - using model claude-sonnet-4-6


In [15]:
def ask_anthropic(prompt: str, system: str | None = None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model: str= "claude-sonnet-4-6", output_format: str | None = None) -> None:
    """
    Send a prompt to Anthropic using the streaming Messages API and return
    the complete generated text.

    Args:
        prompt (str):
            The user prompt to send to the model.
        system (str | None, optional):
            Optional system instruction for the model. In the streaming API
            for this SDK version, it is converted into the required list format.
            Defaults to None.
        temperature (float, optional):
            Controls randomness of the response. Defaults to 0.7.
        max_resp_tokens (int, optional):
            Maximum number of tokens in the generated response. Defaults to 800.
        ai_model (str, optional):
            The model name to use. Defaults to "claude-sonnet-4-6".
        output_format (str | None, optional):
            Optional output format for the streaming API, if supported by the
            installed SDK version. Defaults to None.

    Returns: None
    """

    params = {
        "model": ai_model,
        "max_tokens" : max_resp_tokens,
        "temperature": temperature,
        "messages": [
            {'role': 'user', 'content': prompt}
        ],
    }

    if system:
        params["system"] = [{"type": "text", "text": system}]

    if output_format:
        params["output_format"] = output_format
    
    with claude_client.messages.stream(**params) as stream:
        full_text = ""
        for text in stream.text_stream:
            full_text += text
            clear_output(wait=True)
            # display(Markdown(full_text)) # Only for this module need to be in comments.
    
    return full_text

display(Markdown(f"**The function has run**"))

**The function has run**

---
## 1. LLM-as-Judge Pattern

Use a **stronger model** (or the same model with a judge prompt) to score outputs.

```
Prompt A → Model → Response A ─┐
                                ├─→ Judge (LLM) → Score
Rubric ─────────────────────────┘
```

In [13]:
def evaluate_response(question: str, response: str, criteria: str) -> dict:  
    """Use LLM to evaluate this AI response on a scale of 1-10"""
    import json
    
    judge_prompt = f"""Evaluate this AI response on a scale of 1-10.
Question: {question}
Response: {response}

Criteria: {criteria}

Return JSON: {{"score":<1-10>, "reasoning":"<brief explanation>"}}"""
    
    result = ask_anthropic(prompt=judge_prompt, temperature=0.0)

    try:
        # Clean up markdown code blocks if present
        text = result.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip())
    except:
        return {"score": 0, "reasoning": result}
 
# Test it
question = "What is recursion in programming?"
response = ask_open_ai(question)

In [16]:
eval_response = evaluate_response(
    question,
    response,
    criteria= "Accuracy, clarity, use of examples, appropriate length"
)

display(Markdown(f"### Response:\n{response}\n\n ### Evaluation:\n **Score:** {eval_response.get('score', 'N\A')}\n **Reasoning:** {eval_response.get('reasoning', 'N\A')}"))

### Response:
Recursion in programming is a technique where a function calls itself in order to solve a problem. This approach is often used to break down complex problems into simpler, more manageable subproblems. A recursive function typically has two main components:

1. **Base Case**: This is a condition under which the function will stop calling itself, preventing infinite recursion. It provides a simple, direct answer for the simplest instances of the problem.

2. **Recursive Case**: This is where the function calls itself with modified arguments that bring it closer to the base case. This part of the function defines how the problem is divided into smaller subproblems.

### Example of Recursion

A classic example of recursion is calculating the factorial of a number `n`, denoted as `n!`. The factorial of a non-negative integer `n` is the product of all positive integers less than or equal to `n`. The recursive definition can be expressed as:

- Base Case: \(0! = 1\) and \(1! = 1\)
- Recursive Case: \(n! = n \times (n-1)!\) for \(n > 1\)

Here is how you might implement this in Python:

```python
def factorial(n):
    # Base case
    if n == 0 or n == 1:
        return 1
    # Recursive case
    else:
        return n * factorial(n - 1)

# Example usage
print(factorial(5))  # Output: 120
```

### Advantages of Recursion

- **Simplicity**: Recursive solutions can be more straightforward and easier to read than their iterative counterparts, especially for problems like tree traversals or combinatorial problems.
- **Natural Problem Solving**: Many problems, such as those involving tree structures or divide-and-conquer algorithms, are inherently recursive.

### Disadvantages of Recursion

- **Performance**: Recursive functions can lead to high memory usage due to the call stack and may be less efficient than iterative solutions. Each recursive call adds a new layer to the call stack.
- **Risk of Stack Overflow**: If the recursion goes too deep (e.g., too many recursive calls), it can lead to stack overflow errors, especially in languages that do not optimize tail recursion.

In summary, recursion is a powerful programming concept that can simplify the solution of complex problems, but it should be used judiciously, considering its potential downsides.

 ### Evaluation:
 **Score:** 9
 **Reasoning:** The response excels across all four criteria. Accuracy: The explanation is technically correct, properly defining base cases, recursive cases, and the factorial example with accurate math and working Python code. Clarity: The response is well-structured with headers, bullet points, and bold formatting that make it easy to follow. The language is accessible without being overly simplistic. Use of examples: The factorial example is a classic, appropriate choice that clearly illustrates the concept, and the code is clean and functional with expected output shown. Appropriate length: The response is thorough without being excessive, covering the concept, implementation, and trade-offs in a balanced way. Minor deductions: The response could briefly mention real-world use cases like file system traversal or sorting algorithms to strengthen practical context, and it could mention memoization or dynamic programming as solutions to the performance drawback, which would add more depth for intermediate learners.

---
## 2. A/B Testing Prompts

Compare two different prompts on the **same question**:

In [17]:
question = "Explain what Docker is to a junior developer."

# Simple prompt
response_a = ask_open_ai(question)

# Prompt: PCTF
response_b = ask_open_ai(
    question,
    system_content= "You are a senior DevOps enginner. Use real-world analogy. Keep it under 100 words."
)

criteria = "Clarity for beginner, use of analogies, conciseness, actionable information."

eval_a = evaluate_response(question, response_a, criteria)
eval_b = evaluate_response(question, response_b, criteria)

md = f"""### A/B Test Results
 
| | Prompt A (Simple) | Prompt B (PCTF) |
|---|---|---|
| **Score** | {eval_a.get('score', 'N/A')}/10 | {eval_b.get('score', 'N/A')}/10 |
| **Words** | {len(response_a.split())} | {len(response_b.split())} |
 
#### Prompt A Response
{response_a[:300]}{'...' if len(response_a)>300 else ''}
 
#### Prompt B Response
{response_b[:300]}{'...' if len(response_b)>300 else ''}
"""
display(Markdown(md))
 
winner = "A" if eval_a.get('score',0) > eval_b.get('score',0) else "B"
print(f"\n Winner: Prompt {winner}")

### A/B Test Results

| | Prompt A (Simple) | Prompt B (PCTF) |
|---|---|---|
| **Score** | 8/10 | 8/10 |
| **Words** | 408 | 76 |

#### Prompt A Response
Certainly! Docker is a platform that helps developers build, package, and run applications in a consistent and efficient manner. Here’s a simple breakdown of its key concepts:

### 1. **Containers:**
   - At its core, Docker uses something called **containers**. A container is like a lightweight, po...

#### Prompt B Response
Think of Docker like a shipping container for software. Just as shipping containers standardize how goods are packed, stored, and transported, Docker packages applications and their dependencies into a lightweight, portable container. This ensures that your application runs consistently across diffe...



 Winner: Prompt B


In [ ]:
question = "Explain what Docker is to a junior developer."

# Simple prompt
response_a = ask_open_ai(question)

# Prompt: PCTF
response_b = ask_open_ai(
    question,
    system_content=(
        "You are a senior DevOps enginner. Use real-world analogy. Keep it under 100 words."
        "Use a single real-world analogy (shipping containers) to ground the explanation."
        "End with one concrete terminal command that the junior can try right now."
        "Keep your answer under 80 words.")
)

criteria = (
    "Clarity for beginners (0-10)." 
    "Quality of analogies (0-10)." 
    "Conciseness - shorter is better (0-10)." 
    "Actionable next-step the reader can try immediately (0-10)."
    "Score each dimension, then average for the final score."
)

In [20]:
display(Markdown(response_a))

Sure! Docker is a platform that helps developers create, deploy, and run applications in containers. Think of a container as a lightweight, portable package that includes everything needed to run a piece of software—its code, runtime, libraries, and dependencies. This means that when you package your application in a Docker container, it will run the same way on any machine that has Docker installed, regardless of the underlying operating system or environment.

Here are some key concepts to help you understand Docker better:

1. **Containers**: These are isolated environments where your applications run. Each container is like a mini-computer with its own file system, processes, and network interfaces. Because they're isolated, you can run multiple containers on the same machine without them interfering with each other.

2. **Images**: A Docker image is a blueprint for creating containers. It contains everything needed to run an application, including the application code, libraries, and the environment settings. You can think of it as a snapshot of your application at a specific point in time.

3. **Dockerfile**: This is a script that contains a series of instructions on how to build a Docker image. It specifies things like the base image to use, the application code to copy, and any dependencies that need to be installed.

4. **Docker Hub**: This is a cloud-based registry where you can store and share Docker images. You can pull images from Docker Hub to use as a base for your own applications or push your own images to share with others.

5. **Portability**: Since Docker containers encapsulate everything an application needs, you can run the same container on a developer's laptop, a testing server, or in production without worrying about compatibility issues.

6. **Efficiency**: Containers share the host system's kernel, which makes them more lightweight than traditional virtual machines. This means they use fewer resources and can start up more quickly.

By using Docker, you can streamline the development and deployment process, making it easier to manage dependencies and ensure that your applications run consistently across different environments. It's especially useful in team settings, where different developers might be working on different parts of the same application. 

Overall, Docker is a powerful tool that simplifies the development workflow and helps you build and deploy applications more effectively.

In [21]:
display(Markdown(response_b))

Think of Docker like shipping containers for software. Each container holds everything needed to run an application—code, libraries, and dependencies—just like a shipping container holds goods. This ensures your app runs consistently across different environments, like moving a container from a ship to a truck without unpacking it. 

Try running this command to see Docker in action: 

```bash
docker run hello-world
```

In [23]:
def evaluate_response_enchance(question: str, response: str, criteria: str) -> dict:  
    """Use LLM to evaluate this AI response on a scale of 1-10"""
    import json
    
    eval_system = (
        "You are an expert prompt-engineering evaluator.\n"
        "You will receive a QUESTION, a RESPONSE, and CRITERIA.\n\n"
        "Score EACH dimension 1-10, then compute the average as 'score'.\n\n"
        "Return ONLY valid JSON (no markdown fences). Schema:\n"
        '{"score": <float>, "breakdown": {"clarity": <int>, "analogy": <int>, '
        '"conciseness": <int>, "actionable": <int>}, "one_liner": "<= 15 word summary"}'
    )

    eval_prompt = (
        f"QUESTION:\n{question}\n\n"
        f"RESPONSE:\n{response}\n\n"
        f"CRITERIA:\n{criteria}"
    )
    
    raw = ask_anthropic(prompt=eval_prompt, system= eval_system, temperature=0.0)

    try:
        cleaned = raw.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned.removeprefix("```json").removesuffix("```").strip()
        elif cleaned.startswith("```"):
            cleaned = cleaned.removeprefix("```").removesuffix("```").strip()
 
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"score": 0, "breakdown": {}, "one_liner": "⚠ JSON parse error"}


In [24]:
eval_a = evaluate_response_enchance(question, response_a, criteria)
eval_b = evaluate_response_enchance(question, response_b, criteria)
 
words_a = len(response_a.split())
words_b = len(response_b.split())
 
winner = "A" if eval_a.get("score", 0) > eval_b.get("score", 0) else "B"
 
md = f"""
# A/B Test Results
 
## Question
**{question}**
 
| Metric | Prompt A (Simple) | Prompt B (PCTF) |
|---|---:|---:|
| **Overall Score** | {eval_a.get('score', 'N/A')}/10 | {eval_b.get('score', 'N/A')}/10 |
| **Clarity** | {eval_a.get('breakdown', {}).get('clarity', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('clarity', 'N/A')}/10 |
| **Analogy** | {eval_a.get('breakdown', {}).get('analogy', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('analogy', 'N/A')}/10 |
| **Conciseness** | {eval_a.get('breakdown', {}).get('conciseness', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('conciseness', 'N/A')}/10 |
| **Actionable** | {eval_a.get('breakdown', {}).get('actionable', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('actionable', 'N/A')}/10 |
| **Word Count** | {words_a} | {words_b} |
 
## Prompt A Response
{response_a[:500]}{'...' if len(response_a) > 500 else ''}
 
**Judge summary:** {eval_a.get('one_liner', 'N/A')}
 
---
 
## Prompt B Response
{response_b[:500]}{'...' if len(response_b) > 500 else ''}
 
**Judge summary:** {eval_b.get('one_liner', 'N/A')}
 
---
 
# 🏆 Winner: Prompt {winner}
"""
 
display(Markdown(md))


# A/B Test Results

## Question
**Explain what Docker is to a junior developer.**

| Metric | Prompt A (Simple) | Prompt B (PCTF) |
|---|---:|---:|
| **Overall Score** | 6.5/10 | 9.0/10 |
| **Clarity** | 9/10 | 9/10 |
| **Analogy** | 7/10 | 10/10 |
| **Conciseness** | 4/10 | 9/10 |
| **Actionable** | 6/10 | 8/10 |
| **Word Count** | 379 | 62 |

## Prompt A Response
Sure! Docker is a platform that helps developers create, deploy, and run applications in containers. Think of a container as a lightweight, portable package that includes everything needed to run a piece of software—its code, runtime, libraries, and dependencies. This means that when you package your application in a Docker container, it will run the same way on any machine that has Docker installed, regardless of the underlying operating system or environment.

Here are some key concepts to hel...

**Judge summary:** Clear and well-structured but too long and lacks a hands-on next step

---

## Prompt B Response
Think of Docker like shipping containers for software. Each container holds everything needed to run an application—code, libraries, and dependencies—just like a shipping container holds goods. This ensures your app runs consistently across different environments, like moving a container from a ship to a truck without unpacking it. 

Try running this command to see Docker in action: 

```bash
docker run hello-world
```

**Judge summary:** Clear shipping-container analogy with an immediately runnable Docker command.

---

# 🏆 Winner: Prompt B


---
## 3. Batch Evaluation

Test a prompt across **multiple questions** to measure consistency:

---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **LLM-as-Judge** | Use an LLM to score other LLM outputs |
| **A/B testing** | Compare prompt variants on same questions |
| **Batch evaluation** | Test across multiple inputs for consistency |
| **Scoring rubric** | Define clear criteria (accuracy, clarity, etc.) |
| **JSON output** | Have the judge return structured scores |

---
**Next:** `06_output_parsing.ipynb` — Parse and structure LLM outputs reliably